In [ ]:
# 연습문제: 보험료 가격 결정 구조 분석을 위한 데이터셋의 품질 검사
# 배경
# 왜 어떤 사람의 의료보험 청구 비용은 높고, 어떤 사람은 낮을까? (의료비를 좌우하는 진짜 요인은 무엇일까?)
# 의료 비용은 개인 건강 상태뿐 아니라 나이, 성별, 거주지역, 생활습관, 흡연 여부에 따라 체계적으로 달라집니다.
# KEY QUESTION: 흡연이 의료비를 얼마나 바꿀까?

# 흡연 여부는 의료비를 얼마나 크게 변화시킬까?
# 젊은 사람도 흡연자라면 높은 비용을 지불할까?


# insurance 데이터셋으로 다음 3가지를 수행하세요
# 01 데이터 개요 확인

# 다음 페이지의 Insurance 데이터셋 구조를 먼저 살펴봅니다. (1,338 × 7 / 관측치 × 변수)
# 02 데이터 품질 검사

# 결측치·중복·이상치·자료형을 점검해 분석 가능 상태로 정리합니다. (Quality Check)
# 03 기술 통계량 확인

# 평균·분포·범주 빈도를 통해 데이터의 형태를 파악합니다. (Describe)

# 데이터셋 정보
# 개인 의료보험료 데이터 / 관측치 수: 1,338개 / 변수 수: 7개

# 분석 대상·단위: 개인 의료보험 가입자 (1,338명), 1행 = 1인의 의료보험 기록
# 출처: Kaggle – Medical Cost Personal Datasets (mirichoi0218)
# 공간: 미국 4개 지역 (northeast, northwest, southeast, southwest)

# 유형분류변수설명값 범위/구분종속변수의료비charges개인에게 청구된 연간 의료보험료 (USD)약 1,122 ~ 63,770독립변수인구 통계age피보험자의 나이 (세)18 ~ 64독립변수인구 통계sex성별 (범주형)female, male독립변수건강 지표bmi체질량지수 kg/m² (이상 18.5~24.9)약 15.96 ~ 53.13독립변수건강 지표smoker흡연 여부 (범주형)yes, no독립변수가족 구성children보험에 등재된 부양 자녀 수0 ~ 5독립변수지역region미국 거주 지역 (범주형)NE, NW, SE, SW

# [LAB-05] 1. 연습문제 - 보험료 가격 결정 구조 데이터셋 품질 검사

## #01. 준비작업



### 1. 필요한 라이브러리 참조

In [ ]:
import numpy as np 
from jussam import load_data
from pandas import DataFrame, read_excel 

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


In [ ]:
origin = load_data("insurance")
origin.head()

📚 개인의 나이·성별·BMI·흡연 여부·거주 지역 등 기본 건강·인구학적 정보를 바탕으로 의료보험 청구 비용(charges)을 예측하도록 구성된, 선형회귀와 머신러닝 실습에 널리 사용되는 대표적인 보험 비용 데이터셋 (출처: https://www.kaggle.com/datasets/mirichoi0218/insurance)

    field     description
--  --------  --------------------------------------------------------------
 0  age       가입자의 나이(세)
 1  sex       성별 (male, female)
 2  bmi       체질량 지수(Body Mass Index)
 3  children  부양 자녀 수(보험 내 자녀 수)
 4  smoker    흡연 여부 (yes / no)
 5  region    미국 내 거주 지역 (northeast, northwest, southeast, southwest)
 6  charges   의료보험 청구 비용(달러). 예측해야 하는 타깃 변수.



,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.924
1,18,male,33.770,1,no,southeast,1725.552
2,28,male,33.000,3,no,southeast,4449.462
3,33,male,22.705,0,no,northwest,21984.471
4,32,male,28.880,0,no,northwest,3866.855


In [ ]:
origin.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 94.5 KB


In [ ]:
origin.describe()

,age,bmi,children,charges
count,1338.000,1338.000,1338.000,1338.000
mean,39.207,30.663,1.095,13270.422
std,14.050,6.098,1.205,12110.011
min,18.000,15.960,0.000,1121.874
25%,27.000,26.296,0.000,4740.287
50%,39.000,30.400,1.000,9382.033
75%,51.000,34.694,2.000,16639.913
max,64.000,53.130,5.000,63770.428


In [ ]:
df1 = origin.copy()
df1["sex"] = df1["sex"].astype("category")
df1["smoker"] = df1["smoker"].astype("category")
df1["region"] = df1["region"].astype("category")
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   age       1338 non-null   int64   
 1   sex       1338 non-null   category
 2   bmi       1338 non-null   float64 
 3   children  1338 non-null   int64   
 4   smoker    1338 non-null   category
 5   region    1338 non-null   category
 6   charges   1338 non-null   float64 
dtypes: category(3), float64(2), int64(2)
memory usage: 46.0 KB


In [ ]:
dup = df1.duplicated()
dup

0       False
1       False
2       False
3       False
4       False
        ...  
1333    False
1334    False
1335    False
1336    False
1337    False
Length: 1338, dtype: bool

In [ ]:
dup.sum()

np.int64(1)

In [ ]:
df2 = df1.drop_duplicates()
df2.duplicated().sum()

np.int64(0)

In [ ]:
# 추가함 
fields = df2.select_dtypes(include=['category']).columns

for field in fields: 
    display(df2[field].value_counts())

sex
male      675
female    662
Name: count, dtype: int64

smoker
no     1063
yes     274
Name: count, dtype: int64

region
southeast    364
southwest    325
northeast    324
northwest    324
Name: count, dtype: int64

In [ ]:
# 추가함 
fields = df2.select_dtypes(include="number").columns.to_list()
print(fields)

['age', 'bmi', 'children', 'charges']


In [ ]:
# 추가함 
# 연속형 변수의 단위 확인 
minmax = [] # 최소값과 최대값을 저장할 리스트 

for field in fields : 
    min_value = df2[field].min()
    max_value = df2[field].max()
    minmax.append({"min": min_value, "max":max_value})

# 최소값과 최대값을 데이터 프레임으로 변환하여 출력 
minmax_df = DataFrame(minmax, index=fields)
minmax_df

,min,max
age,18.000,64.000
bmi,15.960,53.130
children,0.000,5.000
charges,1121.874,63770.428


In [ ]:
df2['charges'].value_counts()

charges
16884.924    1
1725.552     1
4449.462     1
21984.471    1
3866.855     1
            ..
10600.548    1
2205.981     1
1629.833     1
2007.945     1
29141.360    1
Name: count, Length: 1337, dtype: int64

# 결측치 확인 

In [ ]:
na_count = df2.isna().sum()
na_count

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [ ]:
rows, cols = df2.shape
print(f"rows: {rows}, cols: {cols}")

rows: 1337, cols: 7


In [ ]:
na_ratio = na_count /rows
na_ratio

age        0.000
sex        0.000
bmi        0.000
children   0.000
smoker     0.000
region     0.000
charges    0.000
dtype: float64

In [ ]:
df2.to_excel("insurance_qtcheck.xlsx", index=False)

In [ ]:
import os
print(os.getcwd())  # 현재 작업 디렉토리 확인

c:\py_temp\15_LAB\[LAB-05] 데이터 품질 점검


In [ ]:
origin_qt = load_data(r"C:\py_temp\15_LAB\[LAB-05] 데이터 품질 점검\insurance_qtcheck.xlsx")

C:\py_temp\15_LAB\[LAB-05] 데이터 품질 점검\insurance_qtcheck.xlsx는 존재하지 않는 데이터에 대한 요청입니다.


In [ ]:
import inspect
print(inspect.getsource(load_data))

def _load_data_remote(key: str, local: str | None = None, view_url: bool = False) -> Optional[DataFrame]:
    """키로 지정된 데이터셋을 로드한다.

    Args:
        key (str): 메타데이터에 정의된 데이터 식별자(파일명 또는 별칭)
        local (str, optional): 로컬 메타데이터 경로. None이면 원격(BASE_URL) 사용.
        view_url (bool, optional): URL을 출력할지 여부. Defaults to False.

    Returns:
        DataFrame | None: 성공 시 데이터프레임, 실패 시 None

    Examples:
        ```python
        from jussam import *
        df = load_data('AD_SALES')  # 메타데이터에 해당 키가 있어야 함
        ```
    """
    index = None
    try:
        url, desc, index, metadata = __get_data_url(key, local=local)
    except Exception as e:
        try:
            print(f"\033{str(e)}\033")
        except Exception:
            print(e)
        return

    #print("\033[data]\033", url.replace("\\", "/"))
    #print("\033[desc]\033", desc)
    print(f"\033📚 {desc}\033")

    if view_url:
        print(f"\033🌐 URL: {url}\033")

    df = None

    try:
        df = __get_df(url, inde

In [ ]:
origin_qt = load_data("insurance_qtcheck")

📚 보험 비용 데이터셋의 품질 검사 완료 버전 (출처: 자체 정제)


In [ ]:
df3 = origin_qt.copy()
df3['sex'] = df3['sex'].astype('category')
df3['smoker'] = df3['smoker'].astype('category')
df3['region'] = df3['region'].astype('category')

df3.info()

<class 'pandas.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   age       1337 non-null   int64   
 1   sex       1337 non-null   category
 2   bmi       1337 non-null   float64 
 3   children  1337 non-null   int64   
 4   smoker    1337 non-null   category
 5   region    1337 non-null   category
 6   charges   1337 non-null   float64 
dtypes: category(3), float64(2), int64(2)
memory usage: 45.9 KB


In [ ]:
desc_df = df3.describe().T
desc_df

,count,mean,std,min,25%,50%,75%,max
age,1337.000,39.222,14.044,18.000,27.000,39.000,51.000,64.000
bmi,1337.000,30.663,6.100,15.960,26.290,30.400,34.700,53.130
children,1337.000,1.096,1.206,0.000,0.000,1.000,2.000,5.000
charges,1337.000,13279.121,12110.360,1121.874,4746.344,9386.161,16657.717,63770.428


In [ ]:
cate_desc_df = df3.describe(include='category').T
cate_desc_df

,count,unique,top,freq
sex,1337,2,male,675
smoker,1337,2,no,1063
region,1337,4,southeast,364


In [ ]:
cate_fields = df3.select_dtypes(include='category').columns

for field in cate_fields: 
    vcount =df3[field].value_counts()
    percent = vcount / df3.shape[0]

    df = DataFrame({'count': vcount, 'percent': percent})
    display(df)

,count,percent
sex,,
male,675,0.505
female,662,0.495


,count,percent
smoker,,
no,1063,0.795
yes,274,0.205


,count,percent
region,,
southeast,364,0.272
southwest,325,0.243
northeast,324,0.242
northwest,324,0.242


# 평균 중앙값의 상대 차이율 계산 

In [ ]:
# "평균-중앙값 상대 차이율 = |평균 -중앙값| / 중앙값" 컬럼 추가 
desc_df['rel_diff'] = abs(desc_df['mean'] - desc_df['50%'] ) / desc_df['50%']

# 상대 차이율 의미 컬럼 추가 
conditions = [desc_df['rel_diff'] < 0.1, desc_df['rel_diff'] < 0.5]
choices = ['similar', 'diff']
desc_df['rdiff_flag'] = np.select(conditions, choices, default='large_diff')

# 필요한 컬럼만 선택하여 출력 
desc_df[['mean', '50%', 'rel_diff', 'rdiff_flag']]

## #04. 이상치 신호 감지 

### 1. IQR 이상치 경계값 계산 

In [ ]:
# iqr 
desc_df['iqr'] = desc_df['75%'] - desc_df['25%']    

# 상한 이상치 경계 
desc_df['upper_bound'] = desc_df['75%'] + 1.5 * desc_df['iqr']

# 하한 이상치 경계 
desc_df['lower_bound'] = desc_df['25%'] - 1.5 * desc_df['iqr']

# 필요한 컬럼만 선택하여 출력 
desc_df[['iqr', 'upper_bound', 'lower_bound']]

,iqr,upper_bound,lower_bound
age,24.000,87.000,-9.000
bmi,8.410,47.315,13.675
children,2.000,5.000,-3.000
charges,11911.373,34524.778,-13120.716


### 2. 명목형 변수를 제외한 데이터 프레임 
- 이상치는 연속형 변수에 대한 개념이므로 명목형 변수를 제외한 데이터 프레임 생성 

In [ ]:
# 명목형 변수의 필드명 추출 
cate_fields = df3.select_dtypes(include='category').columns

df4 = df3.drop(columns=cate_fields)
df4.head()

,age,bmi,children,charges
0,19,27.900,0,16884.924
1,18,33.770,1,1725.552
2,28,33.000,3,4449.462
3,33,22.705,0,21984.471
4,32,28.880,0,3866.855


### 3. 상한 이상치 탐지 

In [ ]:
# 상한 이상치 수 
desc_df['upper_outliers'] = ((df4 > desc_df['upper_bound'])).sum()

# 상한 이상치 수 비율 
desc_df['upper_outliers_ratio'] = desc_df['upper_outliers'] / df4.shape[0]

# 필요한 컬럼만 선택하여 출력 
desc_df[['upper_outliers', 'upper_outliers_ratio']]

,upper_outliers,upper_outliers_ratio
age,0,0.000
bmi,9,0.007
children,0,0.000
charges,139,0.104


### 4. 하한 이상치 탐지 

In [ ]:
# 하한 이상치 수 
desc_df['lower_outliers'] = (df4 < desc_df['lower_bound']).sum()

# 하한 이상치 수 비율
desc_df['lower_outliers_ratio'] = desc_df['lower_outliers'] / df4.shape[0]  

# 필요한 컬럼만 선택하여 출력 
desc_df[['lower_outliers', 'lower_outliers_ratio']]

,lower_outliers,lower_outliers_ratio
age,0,0.000
bmi,0,0.000
children,0,0.000
charges,0,0.000


### 5. 전체 이상치 집계 

In [ ]:
# 통합 이상치 수 
desc_df['outliers'] = desc_df['upper_outliers'] + desc_df['lower_outliers']

# 통합 이상치 수 비율 
desc_df['outliers_ratio'] = desc_df['outliers']/ df3.shape[0] 

# 이상치 탐지 결과 확인 
desc_df[['upper_outliers', 'upper_outliers_ratio', 'lower_outliers', 'lower_outliers_ratio', 'outliers', 'outliers_ratio']]

,upper_outliers,upper_outliers_ratio,lower_outliers,lower_outliers_ratio,outliers,outliers_ratio
age,0,0.000,0,0.000,0,0.000
bmi,9,0.007,0,0.000,9,0.007
children,0,0.000,0,0.000,0,0.000
charges,139,0.104,0,0.000,139,0.104


## #05. 비대칭 신호 확인 

### 1. 왜도 점검 

In [ ]:
# 왜도 계산 
desc_df['skew'] = df4.skew()

# 왜도를 통한 분포 형태 해석 
conditions_skew = [(desc_df['skew'] < -0.5), (desc_df['skew'] > 0.5)]
choices_skew = ['lesf tail', 'right tail']
desc_df['skew_interpret'] = np.select(conditions_skew, choices_skew, default='symmetric')

# 필요한 컬럼만 선택하여 출력 
desc_df[['skew', 'skew_interpret']]

,skew,skew_interpret
age,0.055,symmetric
bmi,0.284,symmetric
children,0.937,right tail
charges,1.515,right tail


### 2. 첨도 점검 


In [ ]:
desc_df['kurt'] = df4.kurt()

# 첨도를 통한 분포 형태 해석 
conditions_kurt = [(desc_df['kurt'] < 0), (desc_df['kurt'] > 0)]
choices_kurt = ['platykurtic', 'leptokurtic']
desc_df['kurt_interpret'] = np.select(conditions_kurt, choices_kurt, default='mesokutic')

# 필요한 컬럼만 선택하여 출력 
desc_df[['kurt', 'kurt_interpret']]

,kurt,kurt_interpret
age,-1.244,platykurtic
bmi,-0.053,platykurtic
children,0.201,leptokurtic
charges,1.604,leptokurtic


### 3. 로그 변환 필요성 판단 함수 정의 

In [ ]:
def judge_log_transfrom(skew, kurt): 
    if skew >= 1:                 # 강한 우측 꼬리 분포 
        return "log1p"
    elif skew > 0.5 and kurt > 0: # 우측 꼬리 분포이면서 첨도가 높은 경우 
        return "log1p"
    elif skew <= -1:              # 강한 좌측 꼬리 분포 
        return "reverse_log1p"
    elif skew < -0.5 and kurt > 0: # 좌측 꼬리 분포이면서 첨도가 높은 경우 
        return "reverse_log1p"
    else: 
        return "none"

### 4. 로그 변환 필요성 판정 

In [ ]:
desc_df['log_need'] = desc_df.apply(lambda row: judge_log_transfrom(
    row['skew'], row['kurt']), axis=1)

# 로그 변환 필요성 결과 확인 
desc_df[['skew', 'kurt', 'log_need']]

,skew,kurt,log_need
age,0.055,-1.244,none
bmi,0.284,-0.053,none
children,0.937,0.201,log1p
charges,1.515,1.604,log1p


## #06. 최종 기술 통계량 확인 

### 2. 기술 통계량 표 저장 

In [ ]:
desc_df.to_excel("insurance_qtcheck_desc.xlsx")